# Temporary LSTM-only recovery — English → Vietnamese

Enable Kaggle **Internet** and a **GPU**, then run all cells. Only LSTM +
attention is trained. The merge utility is bundled for compatible older
GitHub checkouts; model/training helpers still load from the repository.
No earlier Kaggle outputs need to be attached: this notebook downloads the
same pinned corpus and checks its fingerprint before any training.

The recipe matches the supplied run: width 192, two encoder/two decoder layers,
eight epochs maximum, FP32, effective batch 32, LR 0.001, seed 42, the same
early stopping, 25,000 training pairs and fixed 512-example validation scoring.
The short memorization diagnostic is advisory and has **no skip path**.

Download the final **merge-ready ZIP** and extract its contents directly into
`kaggle_runs/rnn_lstm_trsf/`. It contains only LSTM-specific folders, so your
completed RNN/Transformer models and shared prepared data are retained.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/NhanPhan090806/NMT_MarianNMT'
REPO_REF = 'main'
SOURCE_SUBDIR = '.'
WORKSPACE = Path("/kaggle/working")
CLONE_ROOT = WORKSPACE / "nmt_source"
OUTPUT_ROOT = WORKSPACE / "nmt_lstm_only"
RESTORE_FROM = None  # extracted prior output root containing checkpoints/results/data/models

if not REPO_URL.strip():
    raise ValueError("Fill in REPO_URL after pushing the source to GitHub.")
if not WORKSPACE.is_dir():
    raise RuntimeError("Run this training interface on Kaggle.")
if not (CLONE_ROOT / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", "--depth", "1", REPO_URL, str(CLONE_ROOT)], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
subprocess.run(["git", "-C", str(CLONE_ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
REPO_ROOT = (CLONE_ROOT / SOURCE_SUBDIR).resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Check SOURCE_SUBDIR.")
print("Source:", REPO_ROOT)
print("Artifacts:", OUTPUT_ROOT)

## Install the same dependencies and load helpers

In [ ]:
if any(name in sys.modules for name in ("transformers", "huggingface_hub", "tokenizers")):
    raise RuntimeError("Restart the Kaggle session before setup; ML packages are already imported.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)], check=True)
import importlib
SOURCE_ROOT = (REPO_ROOT / "src").resolve()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
import nmt_assignment
if Path(nmt_assignment.__file__).resolve().parent != SOURCE_ROOT / "nmt_assignment":
    raise RuntimeError("Helpers came from another checkout; restart the session.")
import torch
from importlib.metadata import version
from packaging.version import Version
if version("transformers") != "5.17.0" or Version(version("huggingface-hub")) < Version("1.23"):
    raise RuntimeError("Setup versions differ from requirements-kaggle.txt; restart the session.")
from nmt_assignment.workflow import restore_artifacts
from qa_assignment.utils import environment_info
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator.")
DEVICE = "cuda:0"
if RESTORE_FROM is not None:
    print(restore_artifacts(RESTORE_FROM, OUTPUT_ROOT))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Loaded helpers:", nmt_assignment.__file__)
print(environment_info(REPO_ROOT))

## Frozen model, optimizer and data settings

In [ ]:
# Frozen settings copied from the supplied run and notebook 1.

REFERENCE_MODEL_CONFIG = {'stage': 'lstm_attention',
 'd_model': 192,
 'encoder_layers': 2,
 'decoder_layers': 2,
 'heads': 4,
 'feedforward_dim': 768,
 'dropout': 0.1,
 'tie_embeddings': False}

REFERENCE_TRAIN_CONFIG = {'epochs': 8,
 'micro_batch_size': 16,
 'accumulation_steps': 2,
 'eval_batch_size': 16,
 'learning_rate': 0.001,
 'weight_decay': 0.01,
 'warmup_fraction': 0.02,
 'gradient_clip': 1.0,
 'precision': 'fp32',
 'seed': 42,
 'sample_order_seed': 2026,
 'save_every_steps': 250,
 'eval_every_steps': 0,
 'keep_step_checkpoints': 2,
 'development_limit': 512,
 'log_every_steps': 25,
 'early_stopping_patience': 3,
 'early_stopping_min_delta': 0.2,
 'early_stopping_min_steps': 0,
 'early_stopping_min_epochs': 3}

REFERENCE_DATA_CONFIG = {'dataset_name': 'Angelectronic/IWSLT15_English_Vietnamese',
 'dataset_revision': '647d179736b0a4f2b62f860ffc962b067505311b',
 'pretrained_name': 'Helsinki-NLP/opus-mt-en-vi',
 'pretrained_revision': '989c9fb9ec63987901022baf0182dcec3e149be6',
 'validation_examples': 5000,
 'max_train_examples': 25000,
 'split_seed': 2026,
 'vocab_size': 8000,
 'max_input_length': 96,
 'max_output_length': 96,
 'max_words': 60,
 'target_prefix': '>>vie<< '}

EXPECTED_DATA_FINGERPRINT = 'ea3f435272ff49b3607e25de12b01bd4dffe3171a8907a96919f173ca76a16a3'

EXPECTED_MODEL_PARAMETERS = 4672256

REFERENCE_SOURCE_REVISION = 'a1465b4d2bbb2a955fce4a498e49f4a8da6ca275'

from dataclasses import asdict
from nmt_assignment.config import DataConfig, ModelConfig, TrainConfig
from nmt_assignment.data import prepare_data
from nmt_assignment.workflow import (resource_pilot, learning_diagnostic, run_experiment,
    collect_results, plot_comparison, archive_translation_outputs)
from qa_assignment.utils import write_json

MODEL_CONFIG = ModelConfig(**REFERENCE_MODEL_CONFIG)
TRAIN_CONFIG = TrainConfig(**REFERENCE_TRAIN_CONFIG)
DATA_CONFIG = DataConfig(**REFERENCE_DATA_CONFIG)
RUN_RESOURCE_PILOT = True
RUN_LEARNING_DIAGNOSTIC = True
DIAGNOSTIC_STEPS = 1000
BENCHMARK_EXAMPLES = 32
print("Model:", asdict(MODEL_CONFIG))
print("Training:", asdict(TRAIN_CONFIG))

# Generated fallback from src/nmt_assignment/workflow.py. Training code stays in the checkout.
from nmt_assignment import workflow as _notebook_workflow
if hasattr(_notebook_workflow, 'archive_stage_outputs'):
    archive_stage_outputs = _notebook_workflow.archive_stage_outputs
else:
    import json
    from pathlib import Path
    from nmt_assignment.config import STAGES

    def archive_stage_outputs(output_root, stage, seed=42):
        """Package one completed stage for extraction into an existing output root."""
        import zipfile
        if stage not in STAGES:
            raise ValueError("Unknown stage for the merge archive.")
        output_root = Path(output_root)
        seed_folder = f"seed_{seed}"
        folders = [output_root / name / stage / seed_folder for name in ("checkpoints", "results", "models")]
        required = [folders[0] / "best.pt", folders[0] / "last.pt", folders[1] / "summary.json",
                    folders[1] / "test_predictions.json", folders[2] / "translation_config.json",
                    folders[2] / "model.safetensors"]
        if any(not path.is_file() for path in required):
            raise FileNotFoundError("The stage is incomplete; no merge-ready archive was created.")
        summary = json.loads((folders[1] / "summary.json").read_text(encoding="utf-8"))
        export = json.loads((folders[2] / "translation_config.json").read_text(encoding="utf-8"))
        if summary["stage"] != stage or export["model"]["stage"] != stage or summary["seed"] != seed or (
                summary["data_fingerprint"] != export["data_fingerprint"]):
            raise ValueError("The selected summary and export do not describe the same stage/data.")
        files = [path for folder in folders for path in folder.rglob("*") if path.is_file()]
        diagnostic = output_root / "results" / "diagnostics" / stage
        files.extend(path for path in diagnostic.rglob("*") if path.is_file())
        pilot = output_root / "results" / "pilots" / f"{stage}.json"
        if pilot.is_file():
            files.append(pilot)
        archive = output_root.parent / f"{stage}_{seed_folder}_merge_ready.zip"
        with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as output:
            for path in sorted(files):
                if not path.name.endswith(".tmp"):
                    output.write(path, arcname=path.relative_to(output_root))
        return archive
    print("Using notebook-bundled archive_stage_outputs; this checkout lacks the optional helper.")

## Prepare exactly the same corpus

The check includes split files and both tokenizers. If the fingerprint differs,
this cell stops instead of training a model that would not match the comparison.

In [ ]:
bundle = prepare_data(OUTPUT_ROOT / "data", DATA_CONFIG)
actual_fingerprint = bundle.manifest["data_fingerprint"]
if actual_fingerprint != EXPECTED_DATA_FINGERPRINT:
    raise ValueError("Prepared data differs from the supplied run. Do not merge these outputs. "
        "Use the original prepared data with identical DATA_CONFIG before retrying.")
print("Verified identical prepared data:", actual_fingerprint)
print("Counts:", bundle.manifest["counts"])
write_json(OUTPUT_ROOT / "results" / MODEL_CONFIG.stage / f"seed_{TRAIN_CONFIG.seed}" / "recovery_recipe.json",
    {"reference_source_revision": REFERENCE_SOURCE_REVISION,
     "expected_data_fingerprint": EXPECTED_DATA_FINGERPRINT,
     "model": asdict(MODEL_CONFIG), "train": asdict(TRAIN_CONFIG), "data": asdict(DATA_CONFIG),
     "purpose": "recover the missing LSTM stage; same training recipe, advisory memorization probe"})

## LSTM resource and learning probes

Probe weights are discarded. Missing chrF 90 reports the actual result and
proceeds to main training; nonfinite losses/gradients still raise an error.

In [ ]:
try:
    if RUN_RESOURCE_PILOT:
        pilot = resource_pilot(bundle, MODEL_CONFIG, TRAIN_CONFIG, DEVICE)
        if pilot["parameters"] != EXPECTED_MODEL_PARAMETERS:
            raise ValueError("LSTM parameter count differs from the original recipe; inspect the model settings.")
        print(pilot)
        write_json(OUTPUT_ROOT / "results" / "pilots" / "lstm_attention.json", pilot)
    if RUN_LEARNING_DIAGNOSTIC:
        diagnostic = learning_diagnostic(bundle, MODEL_CONFIG, OUTPUT_ROOT,
                                        DEVICE, steps=DIAGNOSTIC_STEPS)
        print("Diagnostic chrF:", diagnostic["final"]["chrf"])
        if not diagnostic["overfit_demonstrated"]:
            print("Memorization target not reached in the short probe. Main LSTM training remains enabled.")
finally:
    print("Partial/recovery archive:", archive_translation_outputs(OUTPUT_ROOT))

## Train LSTM, select its best checkpoint, export and evaluate

This cell always calls the LSTM training helper after successful data preparation.
Its learning curve uses the same validation chrF selection and early stopping.
Checkpoints, test predictions, benchmark metrics and the portable export use
the standard `lstm_attention/seed_42/` paths.

In [ ]:
try:
    summary = run_experiment(bundle, MODEL_CONFIG, TRAIN_CONFIG, OUTPUT_ROOT,
        device=DEVICE, repo_root=REPO_ROOT, resume="auto", benchmark_examples=BENCHMARK_EXAMPLES)
    print(summary["test"])
finally:
    print("Full recovery archive:", archive_translation_outputs(OUTPUT_ROOT))

## Inspect results and try a translation

In [ ]:
table = collect_results(OUTPUT_ROOT)
display(table)
figure = plot_comparison(table, OUTPUT_ROOT)
if figure is not None:
    display(figure)
from nmt_assignment.inference import Translator
MODEL_DIR = OUTPUT_ROOT / "models" / "lstm_attention" / f"seed_{TRAIN_CONFIG.seed}"
translator = Translator.from_export(MODEL_DIR, device=DEVICE)
ENGLISH = "Scientists are studying how the climate is changing."
print("EN:", ENGLISH)
print("VI:", translator.translate(ENGLISH))

## Download and merge

Download the merge-ready ZIP below and extract **inside** your local
`kaggle_runs/rnn_lstm_trsf/` folder. Its top level is `checkpoints/`, `results/`
and `models/`; there is no extra `nmt_lstm_only/` enclosing directory.

Your new summary will be `results/lstm_attention/seed_42/summary.json`.
The archive includes LSTM's checkpoints, model/tokenizer, predictions,
learning curves, diagnostics and recipe metadata. It omits shared data and
top-level comparison CSV/plots, avoiding replacement with a LSTM-only table.
The previous combined CSV/plot can be regenerated during the next local analysis.
Save this Kaggle run's console log as `lstm_only.log` alongside your earlier log.

In [ ]:
merge_archive = archive_stage_outputs(OUTPUT_ROOT, "lstm_attention", seed=TRAIN_CONFIG.seed)
print("DOWNLOAD THIS ZIP:", merge_archive)
print("Extract its contents into: kaggle_runs/rnn_lstm_trsf/")
from IPython.display import FileLink
display(FileLink(str(merge_archive)))